# 🧠 Brainclip Complete Runtime — Colab

**One notebook to rule them all.** This sets up the complete Brainclip backend on Google Colab:

| Capability | Stack |
|---|---|
| 🎵 Voice Synthesis (TTS) | Fish S2-Pro via vLLM-Omni or fish-speech |
| 🎥 Video Rendering | Remotion + Headless Chromium + Node.js 18 |
| 🌐 Public API Tunnel | Cloudflare Tunnel (free, no account needed) |
| 🔍 Transcription | Faster-Whisper large-v2 (word-level timestamps) |

---

### How to use

1. **Runtime → Change runtime type → GPU (T4)** — S2-Pro needs ≥10 GB VRAM
2. **Runtime → Run all** — takes ~10 min on first run
3. Copy the printed **Cloudflare Tunnel URL** into Brainclip → Settings → Colab Tunnel URL
4. **Keep this notebook tab open** — closing it kills the server

### Why an isolated virtual environment?

Colab's base Python packages change between image updates. Pinning everything
inside `/content/venv` makes the notebook immune to silent breakage.


---
## ➀ Preflight checks

Verify Python ≥ 3.10 and GPU availability before we invest time in installing.

In [ ]:
import sys, os, shutil
from pathlib import Path

APP_ROOT = Path('/content')
VENV_DIR = APP_ROOT / 'venv'

print(f'Python version : {sys.version}')
print(f'Colab release  : {os.environ.get("COLAB_RELEASE_TAG", "unknown")}')

if sys.version_info < (3, 10):
    raise RuntimeError(
        f'Brainclip requires Python ≥ 3.10, but this Colab has {sys.version}. '
        'Try Runtime → Disconnect and delete runtime, then reconnect.'
    )
print('✅ Python version OK')

# Quick GPU check
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_properties(0).name
        vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
        print(f'GPU            : {gpu_name} ({vram_gb} GB VRAM)')
        if vram_gb < 10:
            print('⚠️  Less than 10 GB VRAM — S2-Pro may run out of memory.')
            print('   Consider switching to T4 or A100: Runtime → Change runtime type')
        else:
            print('✅ GPU VRAM sufficient for S2-Pro')
    else:
        print('❌ NO GPU! Go to Runtime → Change runtime type → GPU (T4)')
        raise SystemExit(1)
except ImportError:
    print('ℹ️  torch not yet installed — GPU will be checked after install')


---
## ➁ System packages: Node.js 18, Chromium, FFmpeg

These are `apt` dependencies needed by Remotion for headless video rendering.
Skips if already installed.

In [ ]:
%%bash
set -e

echo '=== Installing Node.js 18 ==='
if command -v node &>/dev/null && [[ "$(node -v)" == v18* ]]; then
    echo "Node.js $(node -v) already installed — skipping."
else
    curl -fsSL https://deb.nodesource.com/setup_18.x | sudo -E bash - 2>&1 | tail -5
    sudo apt-get install -y nodejs 2>&1 | tail -3
fi

echo '=== Installing Chromium & FFmpeg ==='
sudo apt-get install -y chromium-browser ffmpeg 2>&1 | tail -5

echo ''
echo "✅ node $(node -v)  |  npm $(npm -v)  |  ffmpeg $(ffmpeg -version 2>&1 | head -1 | awk '{print $3}')"


---
## ➂ Clone Brainclip repo + create virtual environment

1. Clone the repo (skip if already present)
2. Create a fresh venv with `--without-pip` (avoids Colab's `ensurepip` errors)
3. Bootstrap pip manually
4. Pin build tools to known-good versions

**All subsequent `pip install` commands target the venv, never the system Python.**

In [ ]:
%%bash
set -e
cd /content

# ── Clone repo if not already present ──
if [ ! -d "notebooks" ] && [ ! -f "notebooks/brainclip_colab_server.py" ]; then
    echo '=== Cloning Brainclip repository ==='
    git clone --depth 1 https://github.com/harshal0704/Brainclip.git temp_repo 2>&1 | tail -3
    shopt -s dotglob
    mv temp_repo/* . 2>/dev/null || true
    rm -rf temp_repo
    echo '✅ Repository cloned'
else
    echo '✅ Repository already present'
fi

# ── Create fresh venv (only if missing) ──
if [ ! -f /content/venv/bin/python ]; then
    echo '=== Creating isolated virtual environment ==='
    rm -rf /content/venv
    python3 -m venv /content/venv --without-pip
    source /content/venv/bin/activate
    curl -sS https://bootstrap.pypa.io/get-pip.py | python 2>&1 | tail -3
    python -m pip install --upgrade 'pip<25' 'setuptools<75' 'wheel<0.46' 2>&1 | tail -5
    echo ''
    echo "✅ venv created at /content/venv"
else
    echo '✅ venv already exists — skipping creation'
fi

source /content/venv/bin/activate
echo "   pip   : $(python -m pip --version | awk '{print $2}')"
echo "   python: $(python --version)"


---
## ➃ Install Python dependencies

Install order matters to avoid dependency conflicts:

1. **PyTorch 2.6.0** — GPU compute foundation
2. **vLLM-Omni** — high-performance inference engine for S2-Pro
3. **fish-speech** — fallback TTS engine
4. **Application packages** — FastAPI, Whisper, audio libs
5. **Remotion npm** — for video rendering

⏱️ **First run:** ~8–12 minutes  |  **Cached re-run:** ~1–2 minutes

| Package | Version | Purpose |
|---|---|---|
| torch | 2.6.0 | GPU compute |
| torchaudio | 2.6.0 | Audio processing |
| vllm-omni | latest | Primary S2-Pro inference |
| fish-speech | latest | Fallback S2-Pro inference |
| fastapi | 0.115.8 | HTTP server |
| faster-whisper | 1.1.1 | Speech transcription |
| nest-asyncio | 1.6.0 | Colab event loop fix |
| pydantic | 2.10.6 | Data validation |

In [ ]:
%%bash
set -e
source /content/venv/bin/activate

echo '══════════════════════════════════════════════════════'
echo '  [1/5] PyTorch 2.6.0 + torchaudio'
echo '══════════════════════════════════════════════════════'
python -m pip install --upgrade \
    'torch==2.6.0' \
    'torchaudio==2.6.0' \
    'huggingface-hub[hf_xet]>=0.30.0' \
    2>&1 | tail -5

echo ''
echo '══════════════════════════════════════════════════════'
echo '  [2/5] vLLM-Omni (primary TTS engine)'
echo '══════════════════════════════════════════════════════'
python -m pip install vllm-omni 2>&1 | tail -5

echo ''
echo '══════════════════════════════════════════════════════'
echo '  [3/5] fish-speech (fallback TTS engine)'
echo '══════════════════════════════════════════════════════'
python -m pip install fish-speech 2>&1 | tail -5

echo ''
echo '══════════════════════════════════════════════════════'
echo '  [4/5] Application requirements'
echo '══════════════════════════════════════════════════════'
cat > /tmp/brainclip_requirements.txt << 'REQS'
fastapi==0.115.8
uvicorn[standard]==0.34.0
python-multipart==0.0.20
requests==2.32.3
soundfile==0.13.1
numpy==2.1.3
scipy==1.15.2
faster-whisper==1.1.1
sentencepiece==0.2.0
einops==0.8.1
loguru==0.7.3
accelerate==1.2.1
transformers==4.49.0
pydantic==2.10.6
nest-asyncio==1.6.0
REQS
python -m pip install -r /tmp/brainclip_requirements.txt 2>&1 | tail -10

echo ''
echo '══════════════════════════════════════════════════════'
echo '  [5/5] Remotion npm dependencies'
echo '══════════════════════════════════════════════════════'
# Find remotion directory
if [ -d /content/remotion ]; then
    cd /content/remotion && npm install 2>&1 | tail -5
    echo "✅ Remotion installed at /content/remotion"
elif [ -d /content/Brainclip/remotion ]; then
    cd /content/Brainclip/remotion && npm install 2>&1 | tail -5
    echo "✅ Remotion installed at /content/Brainclip/remotion"
else
    echo '⚠️  Remotion directory not found — render will fail unless repo is cloned'
fi

echo ''
echo '══════════════════════════════════════════════════════'
echo '  Package verification'
echo '══════════════════════════════════════════════════════'
python -c "
import importlib, sys
packages = [
    ('torch',          'torch'),
    ('torchaudio',     'torchaudio'),
    ('vllm_omni',      'vllm_omni'),
    ('fish_speech',    'fish_speech'),
    ('fastapi',        'fastapi'),
    ('uvicorn',        'uvicorn'),
    ('numpy',          'numpy'),
    ('scipy',          'scipy'),
    ('pydantic',       'pydantic'),
    ('faster_whisper',  'faster_whisper'),
    ('transformers',   'transformers'),
    ('accelerate',     'accelerate'),
    ('soundfile',      'soundfile'),
    ('nest_asyncio',   'nest_asyncio'),
    ('loguru',         'loguru'),
]
all_ok = True
tts_ok = False
for name, mod in packages:
    try:
        m = importlib.import_module(mod)
        v = getattr(m, '__version__', '✓')
        print(f'  ✅ {name:20s} {v}')
        if name in ('vllm_omni', 'fish_speech'):
            tts_ok = True
    except ImportError:
        # vllm_omni and fish_speech are alternates — only one needed
        if name in ('vllm_omni', 'fish_speech'):
            print(f'  ⚪ {name:20s} not installed (optional)')
        else:
            print(f'  ❌ {name:20s} MISSING')
            all_ok = False

print()
if not tts_ok:
    print('❌ CRITICAL: Neither vllm-omni nor fish-speech is installed!')
    print('   Voice synthesis will not work.')
    sys.exit(1)
elif all_ok:
    print('✅ All dependencies verified')
else:
    print('⚠️  Some packages missing — check errors above')
"


---
## ➄ Download Fish S2-Pro model (~9 GB)

Downloads the sharded S2-Pro checkpoint from HuggingFace.

- **First run:** 5–15 minutes depending on bandwidth
- **Subsequent runs:** instant (cached on disk)

In [ ]:
%%bash
set -e
source /content/venv/bin/activate

MODEL_DIR="/content/models/s2-pro"

# Skip if already downloaded (>5 files = config + weights + tokenizer)
if [ -d "$MODEL_DIR" ] && [ "$(ls -1 $MODEL_DIR 2>/dev/null | wc -l)" -gt 5 ]; then
    echo "✅ Model already present — skipping download."
    du -sh "$MODEL_DIR"
else
    echo '=== Downloading fishaudio/s2-pro ==='
    echo 'This will take 5–15 minutes on first run...'
    mkdir -p "$MODEL_DIR"
    huggingface-cli download fishaudio/s2-pro \
        --local-dir "$MODEL_DIR" \
        --local-dir-use-symlinks False
    echo ''
    echo '✅ Download complete.'
    du -sh "$MODEL_DIR"
fi
echo ''
ls -lh "$MODEL_DIR" | head -15


---
## ➅ Verify GPU + VRAM

Final sanity check before starting the server.

In [ ]:
%%bash
set -e
source /content/venv/bin/activate

python << 'PYEOF'
import torch

print(f'CUDA available : {torch.cuda.is_available()}')
print(f'PyTorch version: {torch.__version__}')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    mem_total = props.total_memory / 1e9
    mem_free, _ = torch.cuda.mem_get_info()
    mem_free_gb = mem_free / 1e9

    print(f'GPU            : {props.name}')
    print(f'VRAM total     : {mem_total:.1f} GB')
    print(f'VRAM free      : {mem_free_gb:.1f} GB')

    if mem_free_gb < 10:
        print('\n⚠️  WARNING: Less than 10 GB free VRAM. S2-Pro may OOM.')
        print('   Try: Runtime → Disconnect and delete runtime → Reconnect with T4/A100')
    else:
        print('\n✅ Sufficient VRAM for S2-Pro')

    # Also verify TTS engine imports
    tts_engine = 'none'
    try:
        import vllm_omni
        tts_engine = 'vllm-omni'
    except ImportError:
        try:
            import fish_speech
            tts_engine = 'fish-speech'
        except ImportError:
            pass
    print(f'TTS engine     : {tts_engine}')
    if tts_engine == 'none':
        print('❌ No TTS engine! Voice synthesis will fail.')
        raise SystemExit(1)
else:
    print('\n❌ NO GPU DETECTED!')
    print('   Go to Runtime → Change runtime type → GPU (T4)')
    raise SystemExit(1)
PYEOF


---
## ➆ Verify server module

Confirm `brainclip_colab_server.py` exists and list its endpoints.

In [ ]:
%%bash
set -e
mkdir -p /content/notebooks

if [ -f /content/notebooks/brainclip_colab_server.py ]; then
    echo '✅ Server module found'
    echo "   Lines: $(wc -l < /content/notebooks/brainclip_colab_server.py)"
else
    echo '❌ Server module not found!'
    echo '   Re-run cell ➂ to clone the repo, or check the clone step.'
    exit 1
fi

# Ensure __init__.py for Python imports
touch /content/notebooks/__init__.py

echo ''
echo '=== Registered endpoints ==='
grep -E '(@app\.(get|post|delete|put))' /content/notebooks/brainclip_colab_server.py | sed 's/^/  /'


---
## ➇ Start FastAPI server

Launches uvicorn on port 8000 using the **venv Python directly** for reliable imports.
The server auto-detects the TTS engine and reports status on `/health`.

In [ ]:
%%bash
set -e
source /content/venv/bin/activate
export BRAINCLIP_APP_ROOT=/content
export PYTHONPATH=/content:$PYTHONPATH

# Kill any previous server
pkill -f 'uvicorn.*brainclip_colab_server' 2>/dev/null || true
sleep 2

# Quick import sanity check
echo '=== Pre-launch import check ==='
/content/venv/bin/python -c "
import torch; print(f'  PyTorch  : {torch.__version__}')
import fastapi; print(f'  FastAPI  : {fastapi.__version__}')
try:
    import vllm_omni; print(f'  vLLM-Omni: ✅')
except ImportError:
    print('  vLLM-Omni: ⚪ not installed')
try:
    import fish_speech; print(f'  fish-speech: ✅')
except ImportError:
    print('  fish-speech: ⚪ not installed')
import nest_asyncio; print(f'  nest-asyncio: ✅')
" || { echo '❌ Import check failed! Re-run the install step.'; exit 1; }

echo ''
echo '=== Starting uvicorn server ==='
nohup /content/venv/bin/python -m uvicorn notebooks.brainclip_colab_server:app \
    --app-dir /content \
    --host 0.0.0.0 \
    --port 8000 \
    > /content/uvicorn.log 2>&1 &

# Wait for server to boot (up to 30 seconds for model loading)
echo 'Waiting for server to start...'
for i in $(seq 1 30); do
    if curl -s http://127.0.0.1:8000/health > /dev/null 2>&1; then
        echo ''
        echo '✅ Server is UP!'
        echo ''
        curl -s http://127.0.0.1:8000/health | python -m json.tool
        exit 0
    fi
    sleep 1
    printf '.'
done

echo ''
echo '❌ Server failed to start within 30 seconds.'
echo ''
echo '=== Last 40 lines of uvicorn.log ==='
tail -40 /content/uvicorn.log
exit 1


---
## ➈ Start Cloudflare Tunnel

Exposes the local server to the internet via a free Cloudflare quick tunnel.

**→ Copy the printed URL into Brainclip → Settings → Colab Tunnel URL**

> ⚠️ The tunnel URL changes every time you restart this cell. Update your settings accordingly.

In [ ]:
%%bash
set -e

# Install cloudflared if not present
if ! command -v cloudflared &>/dev/null; then
    echo '=== Installing cloudflared ==='
    wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
        -O /usr/local/bin/cloudflared
    chmod +x /usr/local/bin/cloudflared
    echo "✅ cloudflared $(cloudflared --version 2>&1 | head -1)"
else
    echo "✅ cloudflared already installed"
fi

# Kill any existing tunnel
pkill -f 'cloudflared tunnel' 2>/dev/null || true
sleep 2

# Start tunnel in background
echo '=== Starting Cloudflare Tunnel ==='
cloudflared tunnel --url http://127.0.0.1:8000 \
    --logfile /content/cloudflared.log \
    --metrics 0.0.0.0:9100 \
    > /content/cloudflared.out 2>&1 &

# Wait for tunnel URL
echo 'Waiting for tunnel URL...'
for i in $(seq 1 25); do
    TUNNEL_URL=$(grep -o 'https://[-a-zA-Z0-9.]*trycloudflare.com' /content/cloudflared.out 2>/dev/null | head -n 1)
    if [ -n "$TUNNEL_URL" ]; then
        break
    fi
    sleep 1
    printf '.'
done

echo ''

if [ -z "$TUNNEL_URL" ]; then
    echo '❌ Tunnel URL not found. Last 20 lines of log:'
    tail -20 /content/cloudflared.out
    exit 1
fi

echo ''
echo '════════════════════════════════════════════════════════════'
echo ''
echo "  🚀 YOUR TUNNEL URL:"
echo ''
echo "  $TUNNEL_URL"
echo ''
echo '════════════════════════════════════════════════════════════'
echo ''
echo '  📋 Paste this URL into:'
echo '     Brainclip → Settings → 🔌 Colab Setup → Colab Tunnel URL'
echo ''
echo '  Also set Render Provider to "Colab" in the 🎬 Render tab'
echo '  and TTS Provider to "Colab" in the 🎙️ TTS Config tab'
echo ''

# Verify tunnel is live
echo '=== Health check via tunnel ==='
curl -s "${TUNNEL_URL}/health" | python -m json.tool || echo '⚠️  Health check failed — wait 10s and try the URL manually'


---
## ➉ Reference audio cache

View cached speaker reference tokens (for voice cloning).

In [ ]:
from pathlib import Path

REF_DIR = Path('/content/cache/refs')
REF_DIR.mkdir(parents=True, exist_ok=True)

files = list(REF_DIR.glob('*'))
print(f'Reference cache: {REF_DIR}')
print(f'Cached files   : {len(files)}')
for f in files:
    size_kb = f.stat().st_size / 1024
    print(f'  - {f.name}  ({size_kb:.1f} KB)')

if not files:
    print('  (empty — references will be cached after first voice job)')


---
## 🛠️ Utilities

Run these cells on-demand for debugging and maintenance.

### 📋 View server logs

In [ ]:
# Last 50 lines of the server log
!tail -50 /content/uvicorn.log


### 🔄 Restart server (without re-installing)

In [ ]:
%%bash
set -e
export BRAINCLIP_APP_ROOT=/content
export PYTHONPATH=/content:$PYTHONPATH

pkill -f 'uvicorn.*brainclip_colab_server' 2>/dev/null || true
sleep 2

nohup /content/venv/bin/python -m uvicorn notebooks.brainclip_colab_server:app \
    --app-dir /content \
    --host 0.0.0.0 \
    --port 8000 \
    > /content/uvicorn.log 2>&1 &

sleep 5
curl -s http://127.0.0.1:8000/health | python -m json.tool && echo '✅ Server restarted' || echo '❌ Failed'


### 🖥️ Check GPU memory

In [ ]:
!nvidia-smi


### 🏥 Test health endpoint

In [ ]:
import requests, json

try:
    r = requests.get('http://127.0.0.1:8000/health', timeout=5)
    data = r.json()
    print(json.dumps(data, indent=2))

    status = '✅' if data.get('status') == 'ok' else '⚠️'
    print(f'\n{status} Server status : {data.get("status")}')
    print(f'   Engine       : {data.get("engine_type")}')
    print(f'   GPU VRAM free: {data.get("gpu_mem_free_gb")} GB')
    print(f'   Model loaded : {data.get("models_loaded")}')
    print(f'   Remotion     : {"ready" if data.get("remotion_ready") else "not installed"}')
except Exception as e:
    print(f'❌ Cannot reach server: {e}')


### 🧹 Clear reference cache

In [ ]:
import requests
try:
    r = requests.delete('http://127.0.0.1:8000/voice/cache', timeout=5)
    print(r.json())
except Exception as e:
    print(f'Error: {e}')


### 🎬 Test render endpoint

In [ ]:
import requests, json

try:
    r = requests.get('http://127.0.0.1:8000/health', timeout=5)
    data = r.json()
    print(f'Remotion ready  : {data.get("remotion_ready", False)}')
    print(f'Remotion dir    : {data.get("remotion_dir", "unknown")}')
    print(f'Engine type     : {data.get("engine_type", "unknown")}')
    print(f'Models loaded   : {data.get("models_loaded", False)}')
    if data.get('remotion_ready'):
        print('\n✅ Render endpoint is ready to accept jobs')
    else:
        print('\n⚠️  Remotion npm deps not installed — run "npm install" in the remotion dir')
except Exception as e:
    print(f'❌ Cannot reach server: {e}')


---
## 📝 Operational Notes

### Architecture
```
Brainclip Web App  ───>  Cloudflare Tunnel  ───>  FastAPI (port 8000)
                                                    ├─ /health            → System status
                                                    ├─ /voice/job         → Fish S2-Pro TTS
                                                    ├─ /voice/encode-ref  → Reference encoding
                                                    ├─ /voice/clone       → Voice cloning
                                                    ├─ /render            → Remotion video render
                                                    └─ /render/job/{id}   → Render progress
```

### Settings configuration

After getting the tunnel URL, configure these in Brainclip Settings:
1. **🔌 Colab Setup** → Paste Tunnel URL → Click Test
2. **🎙️ TTS Config** → Set TTS Provider to **Colab Server**
3. **🎬 Render** → Set Render Provider to **Colab Server**
4. Click **Save Settings**

### Common issues

| Issue | Solution |
|---|---|
| `model_oom` error | Runtime → Disconnect and delete runtime → Run All |
| Tunnel URL stops working | Re-run cell ➈ only |
| Server not responding | Check logs: run the utility cell |
| `No GPU detected` | Runtime → Change runtime type → GPU (T4) |
| Stale venv after Colab update | `!rm -rf /content/venv` then re-run from cell ➂ |
| `engine_type: none` | Re-run install cell ➃ — check vllm-omni/fish-speech output |

### File layout
```
/content/
├─ venv/                       ← Isolated Python environment
├─ models/s2-pro/              ← Fish S2-Pro model weights (~9 GB)
├─ cache/refs/                 ← Cached speaker reference tokens
├─ render_work/                ← Temporary render I/O
├─ notebooks/
│  └─ brainclip_colab_server.py ← FastAPI server module
├─ remotion/                   ← Remotion video templates
├─ uvicorn.log                 ← Server log
└─ cloudflared.out             ← Tunnel log
```

### Tips
- **Don't close this tab** — the server dies when the Colab session ends
- Models stay loaded in GPU memory across jobs (no re-load per request)
- If GPU memory drops below 2 GB free, restart the runtime
- The tunnel URL changes each restart — update Brainclip settings
- Use the utility cells below for quick debugging